In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================================================
# 1. CARREGAR DATASET_FEATURES_V4
# ============================================================
df = pd.read_parquet('../data/gold/dataset_features_v4.parquet')
df['data'] = pd.to_datetime(df['data'])
df = df.sort_values('data').reset_index(drop=True)

print(f"dataset_features_v4 carregado: {df.shape}")
print(f"Período: {df['data'].min().date()} → {df['data'].max().date()}")
print(f"\nNovas features vs v3:")
features_v3 = ['fator_nowcasting', 'casos_nowcast', 'municipio_id',
               'casos_por_100k', 'casos_nowcast_por_100k']
features_trends = ['trends_dengue', 'trends_lag_7d', 'trends_lag_14d', 'trends_lag_21d']
features_ndbi = ['ndbi_gee', 'ndbi_lag_30d', 'ndbi_lag_60d']

print(f"  Nowcasting: {features_v3}")
print(f"  Google Trends: {features_trends}")
print(f"  NDBI: {features_ndbi}")

# Verificar nulos
nulos = df.isnull().sum()
nulos = nulos[nulos > 0]
print(f"\nColunas com nulos: {len(nulos)}")
if len(nulos) > 0:
    print(nulos)

dataset_features_v4 carregado: (2242, 67)
Período: 2018-02-12 → 2024-12-28

Novas features vs v3:
  Nowcasting: ['fator_nowcasting', 'casos_nowcast', 'municipio_id', 'casos_por_100k', 'casos_nowcast_por_100k']
  Google Trends: ['trends_dengue', 'trends_lag_7d', 'trends_lag_14d', 'trends_lag_21d']
  NDBI: ['ndbi_gee', 'ndbi_lag_30d', 'ndbi_lag_60d']

Colunas com nulos: 7
radiacao_lag_28d    28
radiacao_mm_14d     13
trends_lag_7d        7
trends_lag_14d      14
trends_lag_21d      21
ndbi_lag_30d        30
ndbi_lag_60d        60
dtype: int64


In [2]:
# ============================================================
# 2. PREPARAR FEATURES — remover leakage + tratar nulos
# ============================================================

# Remover nulos
df_clean = df.dropna().copy()
print(f"Após remover nulos: {df_clean.shape} (removidos {len(df)-len(df_clean)})")

# Remover leakage (mesma decisão do v3)
leakage_cols = ['casos_por_100k', 'casos_nowcast_por_100k', 
                'fator_nowcasting', 'trends_dengue']  # trends lag=0 é leakage
drop_cols = ['data', 'casos', 'casos_nowcast', 'municipio_id'] + leakage_cols

X = df_clean.drop(columns=drop_cols)
y = df_clean['casos']

print(f"Features para treino: {X.shape[1]}")
print(f"Target: casos (min={y.min()}, max={y.max()}, média={y.mean():.1f})")
print(f"\nFeatures de Trends (sem leakage):")
trends_feats = [c for c in X.columns if 'trends' in c]
print(f"  {trends_feats}")
print(f"\nFeatures NDBI:")
ndbi_feats = [c for c in X.columns if 'ndbi' in c]
print(f"  {ndbi_feats}")

# Comparação de features
print(f"\nEvolução do dataset:")
print(f"  v2: 51 features")
print(f"  v3: 53 features (+nowcasting)")
print(f"  v4: {X.shape[1]} features (+trends +ndbi)")

Após remover nulos: (2182, 67) (removidos 60)
Features para treino: 59
Target: casos (min=1, max=440, média=67.3)

Features de Trends (sem leakage):
  ['trends_lag_7d', 'trends_lag_14d', 'trends_lag_21d']

Features NDBI:
  ['ndbi_gee', 'ndbi_lag_30d', 'ndbi_lag_60d']

Evolução do dataset:
  v2: 51 features
  v3: 53 features (+nowcasting)
  v4: 59 features (+trends +ndbi)


In [3]:
# ============================================================
# 3. RETREINO LIGHTGBM V4 COM OPTUNA
# ============================================================
tscv = TimeSeriesSplit(n_splits=5)

def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denom > 0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

# Otimização Optuna — Fold 4
fold_idx = list(tscv.split(X))[3]
X_tr, X_vl = X.iloc[fold_idx[0]], X.iloc[fold_idx[1]]
y_tr, y_vl = y.iloc[fold_idx[0]], y.iloc[fold_idx[1]]

def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': 42
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_vl, y_vl)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    return mean_absolute_error(y_vl, np.maximum(model.predict(X_vl), 0))

print("Otimizando hiperparâmetros v4 (50 trials)...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)
print(f"Melhor MAE Fold 4: {study.best_value:.2f}")

# Retreino completo com melhores params
best_params = {
    'objective': 'regression', 'metric': 'mae',
    'verbosity': -1, 'boosting_type': 'gbdt',
    'n_estimators': 1000, 'random_state': 42,
    **study.best_params
}

resultados = []
modelos = []

print("\nTreinando LightGBM v4 — TimeSeriesSplit 5 folds...")
print("="*60)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMRegressor(**best_params)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    
    y_pred = np.maximum(model.predict(X_val), 0)
    
    mae   = mean_absolute_error(y_val, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_val, y_pred))
    r2    = r2_score(y_val, y_pred)
    sm    = smape(y_val, y_pred)
    
    resultados.append({'fold': fold, 'mae': mae, 'rmse': rmse, 'r2': r2, 'smape': sm})
    modelos.append(model)
    print(f"Fold {fold}: MAE={mae:.1f} | RMSE={rmse:.1f} | R²={r2:.3f} | sMAPE={sm:.1f}%")

res = pd.DataFrame(resultados)
res25 = res[res['fold'] > 1]

print("="*60)
print(f"\nLightGBM v4 — Folds 2-5:")
print(f"  MAE:   {res25['mae'].mean():.1f} ± {res25['mae'].std():.1f}")
print(f"  RMSE:  {res25['rmse'].mean():.1f} ± {res25['rmse'].std():.1f}")
print(f"  R²:    {res25['r2'].mean():.3f} ± {res25['r2'].std():.3f}")
print(f"  sMAPE: {res25['smape'].mean():.1f} ± {res25['smape'].std():.1f}%")

print(f"\nEvolução dos modelos:")
print(f"{'Modelo':<20} {'MAE':>8} {'RMSE':>8} {'R²':>8} {'sMAPE':>8}")
print(f"{'-'*55}")
print(f"{'LightGBM v2':<20} {'17.4':>8} {'27.8':>8} {'0.830':>8} {'N/A':>8}")
print(f"{'LightGBM v3':<20} {'17.5':>8} {'27.9':>8} {'0.829':>8} {'32.4%':>8}")
print(f"{'LightGBM v4':<20} {res25['mae'].mean():>8.1f} {res25['rmse'].mean():>8.1f} {res25['r2'].mean():>8.3f} {res25['smape'].mean():>7.1f}%")

Otimizando hiperparâmetros v4 (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

Melhor MAE Fold 4: 15.64

Treinando LightGBM v4 — TimeSeriesSplit 5 folds...
Fold 1: MAE=51.7 | RMSE=86.8 | R²=0.122 | sMAPE=57.1%
Fold 2: MAE=11.9 | RMSE=18.0 | R²=0.750 | sMAPE=34.9%
Fold 3: MAE=18.4 | RMSE=28.2 | R²=0.814 | sMAPE=36.0%
Fold 4: MAE=15.6 | RMSE=26.6 | R²=0.852 | sMAPE=25.5%
Fold 5: MAE=24.5 | RMSE=40.6 | R²=0.865 | sMAPE=29.6%

LightGBM v4 — Folds 2-5:
  MAE:   17.6 ± 5.3
  RMSE:  28.4 ± 9.3
  R²:    0.820 ± 0.052
  sMAPE: 31.5 ± 4.9%

Evolução dos modelos:
Modelo                    MAE     RMSE       R²    sMAPE
-------------------------------------------------------
LightGBM v2              17.4     27.8    0.830      N/A
LightGBM v3              17.5     27.9    0.829    32.4%
LightGBM v4              17.6     28.4    0.820    31.5%


In [4]:
# ============================================================
# 4. SALVAR MODELO V4 + MÉTRICAS
# ============================================================
import json
from pathlib import Path

Path('../models').mkdir(exist_ok=True)

# Salvar modelo do Fold 5 (maior treino)
modelo_v4 = modelos[4]
joblib.dump(modelo_v4, '../models/lgbm_v4_producao.pkl')

# Salvar métricas
res.to_csv('../reports/metricas_lgbm_v4.csv', index=False)

# Resumo JSON
resumo = {
    'modelo': 'LightGBM v4 (Optuna)',
    'dataset': 'dataset_features_v4',
    'n_features': int(X.shape[1]),
    'n_registros': int(len(df_clean)),
    'periodo': '2018-02-12 a 2024-12-28',
    'validacao': 'TimeSeriesSplit 5 folds (Folds 2-5)',
    'metricas': {
        'MAE': round(res25['mae'].mean(), 1),
        'MAE_dp': round(res25['mae'].std(), 1),
        'RMSE': round(res25['rmse'].mean(), 1),
        'RMSE_dp': round(res25['rmse'].std(), 1),
        'R2': round(res25['r2'].mean(), 3),
        'R2_dp': round(res25['r2'].std(), 3),
        'sMAPE': round(res25['smape'].mean(), 1),
        'sMAPE_dp': round(res25['smape'].std(), 1),
    },
    'comparacao': {
        'v2_R2': 0.830, 'v3_R2': 0.829, 'v4_R2': round(res25['r2'].mean(), 3),
        'conclusao': 'Google Trends e NDBI nao melhoram predicao historica — valor em producao'
    },
    'best_params': study.best_params
}

with open('../reports/resumo_metricas_v4.json', 'w', encoding='utf-8') as f:
    json.dump(resumo, f, ensure_ascii=False, indent=2)

print("Modelo salvo: models/lgbm_v4_producao.pkl")
print("Métricas: reports/metricas_lgbm_v4.csv")
print("Resumo: reports/resumo_metricas_v4.json")
print(f"\nModelo de produção atualizado: LightGBM v4")
print(f"   MAE:   {res25['mae'].mean():.1f} ± {res25['mae'].std():.1f} casos/dia")
print(f"   RMSE:  {res25['rmse'].mean():.1f} ± {res25['rmse'].std():.1f} casos/dia")
print(f"   R²:    {res25['r2'].mean():.3f} ± {res25['r2'].std():.3f}")
print(f"   sMAPE: {res25['smape'].mean():.1f} ± {res25['smape'].std():.1f}%")

Modelo salvo: models/lgbm_v4_producao.pkl
Métricas: reports/metricas_lgbm_v4.csv
Resumo: reports/resumo_metricas_v4.json

Modelo de produção atualizado: LightGBM v4
   MAE:   17.6 ± 5.3 casos/dia
   RMSE:  28.4 ± 9.3 casos/dia
   R²:    0.820 ± 0.052
   sMAPE: 31.5 ± 4.9%
